## module 1 체감온도

### 변수선언

In [11]:
import numpy as np
from datetime import datetime
import pandas as pd

ta = 36 #기온
rh = 50 #상대습도 %그대로 적으면 됨
v =  5.76 #풍속 km/h 
user_types = ['어린이', '농촌'] #사용자의 환경 입력(노인, 어린이, 취약 거주환경, 농촌, 비닐하우스)
                              #[실외작업장_도로, 실외작업장_건설현장, 실외작업장_조선소] --> 빼는게 나을 듯)
result = 0 #가중치 초기화

#습구온도
tw = ta*np.arctan(0.151977*(rh+8.313659)**(1/2))+np.arctan(ta+rh)\
    -np.arctan(rh-1.67633)+(0.00391838*rh**(3/2))*np.arctan(0.023101*rh)-4.686035



### 사용자의 환경별 가중치 함수 정의
###### 시간대는 12~15시를 12시로 간주 15~18시를 15시로 간주 이런식으로 코딩

In [12]:
def weh(user_types, time_slot):
    
    # 시간대별 가중치 정의
    weights = {
        '어린이': {
            3: 0.0, 6: 0.0, 9: 2.9, 12: 2.5, 15: 1.2, 18: -0.9, 21: 0.0, 24: 0.0
        },
        '취약 거주환경': {
            3: 1.0, 6: 0.0, 9: 0.7, 12: 1.7, 15: 0.9, 18: 0.3, 21: 5.0, 24: 2.5
        },
        '농촌': {
            3: 0.0, 6: 0.0, 9: 0.1, 12: 0.5, 15: 1.1, 18: 0.6, 21: 0.0, 24: 0.0
        },
        '비닐하우스': {
            3: 0.0, 6: 0.0, 9: 2.3, 12: 4.0, 15: 4.0, 18: 2.1, 21: 1.0, 24: 0.5
        },
        '실외작업장_도로': {
            3: 0.3, 6: 0.5, 9: 1.2, 12: 1.4, 15: 1.3, 18: 0.9, 21: 0.5, 24: 0.3
        },
        '실외작업장_건설현장': {
            3: 1.2, 6: 1.0, 9: 1.3, 12: 1.6, 15: 1.2, 18: 0.7, 21: 1.4, 24: 1.2
        },
        '실외작업장_조선소': {
            3: 0.0, 6: 0.0, 9: 1.7, 12: 2.2, 15: 1.0, 18: 0.0, 21: 0.0, 24: 0.0
        }
    }

    # 선택된 조건들만 가중치 합산
    total_weight = 0
    for user_type in user_types:
        if user_type in weights:
            total_weight += weights[user_type].get(time_slot, 0)
    
    return total_weight

# 현재 시간을 3, 6, 9, ~ 24에 맞도록 변경
def get_time_slot():
    # 현재 시간 가져오기
    current_hour = datetime.now().hour
    
    # 시간대를 기반으로 time_slot 값 결정
    if 0 <= current_hour < 3:   
        return 24
    elif 3 <= current_hour < 6: 
        return 3
    elif 6 <= current_hour < 9: 
        return 6
    elif 9 <= current_hour < 12: 
        return 9
    elif 12 <= current_hour < 15: 
        return 12
    elif 15 <= current_hour < 18: 
        return 15
    elif 18 <= current_hour < 21: 
        return 18
    elif 21 <= current_hour < 24: 
        return 21
    else: 
        return 24

# time_slot에 3, 6, 9, 형태로 현재 시간을 저장
time_slot = get_time_slot()
print(time_slot)

12


### 여름철 체감온도
###### 소책자에 나와있는 수식
###### 5월부터 9월까지는 이 공식 사용용

In [13]:
result = weh(user_types, time_slot) #가중치
print(f"현재 가중치 값: {result}")

#체감온도 계산
summer_eftem = -0.2442 + 0.55399*tw + 0.45535*ta - (0.0022*tw**(2)) + 0.00278*tw*ta + result + 3.5
print(f"현재 체감온도 값: {summer_eftem}")


현재 가중치 값: 3.0
현재 체감온도 값: 38.94765166028464


### 겨울철 체감온도
###### 10월 부터 4월까지는 이 공식 사용
###### 기온 10도 이하, 풍속 1.3m/s 이상일 때만 산출
###### 아닌경우 그냥 온도 그대로 체감온도 출력


In [14]:
def winter_eftem(ta, v):
    # 기온이 10도 이하이고 풍속이 1.3m/s 이상일 때 체감온도 공식 사용
    if ta <= 10 and v/3.6 >= 1.3:  #풍속을 km/h로 입력 받기 때문에 3.6을 나누어서 m/s로 변환
        eftem = 13.12 + 0.6215 * ta - 11.37 * (v**0.16) + (0.3965 * (v**0.16)) * ta
        return round(eftem, 2)  # 체감온도를 소수점 2자리로 반환
    else:
        return ta  # 조건을 충족하지 않으면 기온 그대로 반환

eftem2 = winter_eftem(ta, v)
print(f"겨울철 체감온도 값: {eftem2}")


겨울철 체감온도 값: 36


### 통합된 체감온도

In [15]:
def merge_eftem(ta, v, rh):
    """
    현재 날짜에 따라 겨울 또는 여름 체감온도를 계산.
    - 겨울 (10월 ~ 4월): winter_eftem 공식
    - 여름 (5월 ~ 9월): summer_eftem 공식
    """
    # 현재 월 가져오기
    current_month = datetime.now().month

    if current_month in [10, 11, 12, 1, 2, 3, 4]:  # 겨울 공식 적용
        if ta <= 10 and v >= 1.3:
            eftem = 13.12 + 0.6215 * ta - 11.37 * (v ** 0.16) + (0.3965 * (v ** 0.16)) * ta
            return round(eftem, 2)  # 소수점 2자리로 반환
        else:
            return ta  # 조건 미충족 시 기온 그대로 반환

    elif current_month in [5, 6, 7, 8, 9]:  # 여름 공식 적용
        # 개인별 가중치
        result = weh(user_types, time_slot) #가중치
        
        # 여름 체감온도 공식
        eftem = -0.2442 + 0.55399 * tw + 0.45535 * ta - (0.0022 * tw ** 2) + \
                (0.00278 * tw * ta) + result + 3.5
        return round(eftem, 2)  # 소수점 2자리로 반환
    else:
        raise ValueError("월(month)은 1에서 12 사이의 값이어야 합니다.")
    

apparent_temperature = merge_eftem(ta, v, rh)


print(f"현재: {datetime.now().month}월")
print(f"체감온도: {apparent_temperature}°C")


현재: 4월
체감온도: 36°C


### 노인, 어린이, 취약거주환경인 경우
### 농촌, 비닐하우스인 경우

In [16]:
def check_risk(user_types, eftem):
    # 노인, 어린이, 취약 거주환경 중 하나라도 해당되면 체크
    applicable_types = {'노인', '어린이', '취약거주환경'}
    # 농촌, 비닐하우스 중 하나라도 해당되면 체크
    applicable_types2 = {'농촌', '비닐하우스'}
    
    # 입력된 user_types에 하나라도 노인, 어린이, 취약거주환경이 포함되면 True
    if any(user_type in applicable_types for user_type in user_types):
        if eftem >= 37:
            return "위험"
        elif 34 <= eftem < 37:
            return "경고"
        elif 31 <= eftem < 34:
            return "주의"
        elif 29 <= eftem < 31:
            return "관심"
        else:
            return "안전"
        
    # 입력된 user_type에 하나라도 농촌, 비닐하우스이 포함되면 True
    elif any(user_type in applicable_types2 for user_type in user_types):
        if eftem >= 38:
            return "위험"
        elif 35 <= eftem < 38:
            return "경고"
        elif 33 <= eftem < 35:
            return "주의"
        elif 31 <= eftem < 33:
            return "관심"
        else:
            return "안전"
    
    # 그 어디에도 해당 되지 않는 다면 e.g. 건장한 성인 남성 (이건 다른 기준 보고 수치 다시 조절해야함)
    else:
        if eftem >= 38:
            return "위험"
        elif 35 <= eftem < 38:
            return "경고"
        elif 33 <= eftem < 35:
            return "주의"
        elif 31 <= eftem < 33:
            return "관심"
        else:
            return "안전"
risk = check_risk(user_types, apparent_temperature)
print(f"사용자의 해당 환경: {user_types}")
print(f"사용자 가중치 값: {result}")
print(f"사용자의 체감온도: {apparent_temperature}")
print(f"사용자 위험 상태: {risk}")


사용자의 해당 환경: ['어린이', '농촌']
사용자 가중치 값: 3.0
사용자의 체감온도: 36
사용자 위험 상태: 경고


## module 1 천식, 폐질환 가능 지수

### 변수 선언

In [17]:
mintem = 0 # 최저기온
drhythm = 8 # 일교차
lpressure = 1025.0-10.4 #현지기압
#상대습도는 위에서 정의

### 변수별 점수 정의

In [18]:
def calculate_score(column_name, value):
    # 서울 항목별 점수 기준 정의
    criteria = {
        '최저기온': {
            4: (-float('inf'), -8.1),  # 4점: -8.1 미만
            3: (-8.1, 0.6),           # 3점: 0.6 미만
            2: (0.6, 13.5),           # 2점: 13.5 미만
            1: (13.5, float('inf'))   # 1점: 13.5 이상
        },
        '일교차': {
            4: (12.5, float('inf')),  # 4점: 12.5 이상
            3: (9.9, 12.5),           # 3점: 9.9 이상
            2: (7.3, 9.9),            # 2점: 7.3 이상
            1: (-float('inf'), 7.3)   # 1점: 7.3 미만
        },
        '현지기압': {
            4: (1017.9, float('inf')), # 4점: 1017.9 이상
            3: (1011.9, 1017.9),       # 3점: 1011.9 이상
            2: (1003.5, 1011.9),       # 2점: 1003.5 이상
            1: (-float('inf'), 1003.5) # 1점: 1003.5 미만
        },
        '상대습도': {
            4: (-float('inf'), 37.4),  # 4점: 37.4 미만
            3: (37.4, 50.0),           # 3점: 50.0 미만
            2: (50.0, 65.9),           # 2점: 65.9 미만
            1: (65.9, float('inf'))    # 1점: 65.9 이상
        }
    }
    
    # 해당 열(column_name)의 값(value)에 대한 점수 계산
    for score, (lower, upper) in criteria[column_name].items():
        if lower <= value < upper:
            return score
    return None  # 범위에 맞는 점수가 없을 경우

# 데이터 입력
data = {
    '최저기온': mintem,
    '일교차': drhythm,
    '현지기압': lpressure,
    '상대습도': rh
}

# 각 항목에 대한 점수 계산
scores = {column: calculate_score(column, value) for column, value in data.items()}
print(scores)


{'최저기온': 3, '일교차': 2, '현지기압': 3, '상대습도': 2}


### 천식, 폐질환 가능지수 수식

In [19]:
ALI = 0.443*scores['최저기온'] + 0.202*scores['일교차'] + 0.315*scores['현지기압'] + 0.04*scores['상대습도']
print(f"사용자의 천식, 폐질환 가능지수: {ALI}")

사용자의 천식, 폐질환 가능지수: 2.758


### 천식, 폐질환 가능지수 등급 산출

In [20]:
def check_ali_level(ALI):
    # ALI 지수 범위별 레벨 정의
    if ALI >= 3.0525:
        return "매우 높음"
    elif 2.6452 <= ALI < 3.0525:
        return "높음"
    elif 1.5354 <= ALI < 2.6452:
        return "보통"
    elif 1.0 <= ALI < 1.5354:
        return "낮음"
    else:
        return "잘못된 값"  # 1 미만의 ALI 값 처리 (기준 외)

# 함수 호출
ALI_risk = check_ali_level(ALI)
print(f"ALI 지수: {ALI}")
print(f"등급 값: {ALI_risk}")

ALI 지수: 2.758
등급 값: 높음
